In [1]:
pip install pandas orange3

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 538.0/538.0 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.5/270.5 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.9/125.9 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.4/51.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.0/95.0 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.8/70.8 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.

In [2]:
import pandas as pd
import math

# ============================================================
# DATASET
# ============================================================

data = {
    "Outlook": [
        "Sunny", "Sunny", "Overcast", "Rain",
        "Rain", "Rain", "Overcast", "Sunny",
        "Sunny", "Rain", "Sunny", "Overcast",
        "Overcast", "Rain"
    ],

    "Temperature": [
        "Hot", "Hot", "Hot", "Mild",
        "Cool", "Cool", "Cool", "Mild",
        "Cool", "Mild", "Mild", "Mild",
        "Hot", "Mild"
    ],

    "Humidity": [
        "High", "High", "High", "High",
        "Normal", "Normal", "Normal", "High",
        "Normal", "Normal", "Normal", "High",
        "Normal", "High"
    ],

    "Wind": [
        "Weak", "Strong", "Weak", "Weak",
        "Weak", "Strong", "Strong", "Weak",
        "Weak", "Weak", "Strong", "Strong",
        "Weak", "Strong"
    ],

    "Play": [
        "No", "No", "Yes", "Yes",
        "Yes", "No", "Yes", "No",
        "Yes", "Yes", "Yes", "Yes",
        "Yes", "No"
    ]
}

df = pd.DataFrame(data)

print("================================================")
print("DATASET")
print("================================================")
print(df)
print()


# ============================================================
# CN2 CLASSIFICATION RULE LEARNING
# ============================================================

# Install Orange3 if needed:
# pip install orange3

from Orange.data import Table, Domain, DiscreteVariable
from Orange.classification.rules import CN2Learner


# Create attributes
attributes = []

for column in df.columns[:-1]:
    values = [str(v) for v in df[column].unique()]

    attributes.append(
        DiscreteVariable(
            column,
            values=values
        )
    )


# Create target variable
target_column = df.columns[-1]

target_values = [
    str(v) for v in df[target_column].unique()
]

target = DiscreteVariable(
    target_column,
    values=target_values
)


# Create Orange domain
domain = Domain(
    attributes,
    target
)


# Convert dataframe into Orange Table
orange_data = Table.from_list(
    domain,
    df.astype(str).values.tolist()
)


# Train CN2
learner = CN2Learner()

classifier = learner(orange_data)


# Display CN2 rules
print("================================================")
print("CN2 CLASSIFICATION RULES")
print("================================================")

for rule in classifier.rule_list:
    print(rule)

print()


# ============================================================
# FOIL ALGORITHM
# ============================================================

class FOIL:

    def __init__(self, data, target_column):

        self.data = data
        self.target_column = target_column


    # --------------------------------------------------------
    # FOIL INFORMATION GAIN
    # --------------------------------------------------------

    def foil_gain(self, p, n, p_new, n_new):

        if p == 0 or p_new == 0:
            return 0

        old_probability = p / (p + n)

        new_probability = p_new / (p_new + n_new)

        if old_probability == 0 or new_probability == 0:
            return 0

        return p_new * (
            math.log2(new_probability)
            - math.log2(old_probability)
        )


    # --------------------------------------------------------
    # APPLY A RULE
    # --------------------------------------------------------

    def apply_rule(self, data, rule):

        result = data.copy()

        for feature, value in rule:

            result = result[
                result[feature] == value
            ]

        return result


    # --------------------------------------------------------
    # LEARN ONE RULE
    # --------------------------------------------------------

    def learn_one_rule(
        self,
        positives,
        negatives
    ):

        rule = []

        current_pos = positives.copy()
        current_neg = negatives.copy()

        features = [
            column
            for column in self.data.columns
            if column != self.target_column
        ]


        # Continue adding conditions
        # until no negative examples remain

        while len(current_neg) > 0:

            best_literal = None
            best_score = -float("inf")

            best_pos = None
            best_neg = None


            # Try every possible condition

            for feature in features:

                for value in self.data[feature].unique():

                    literal = (feature, value)


                    # Don't use same condition twice

                    if literal in rule:
                        continue


                    pos_subset = current_pos[
                        current_pos[feature] == value
                    ]

                    neg_subset = current_neg[
                        current_neg[feature] == value
                    ]


                    # Condition must cover
                    # at least one positive example

                    if len(pos_subset) == 0:
                        continue


                    score = self.foil_gain(
                        len(current_pos),
                        len(current_neg),
                        len(pos_subset),
                        len(neg_subset)
                    )


                    if score > best_score:

                        best_score = score

                        best_literal = literal

                        best_pos = pos_subset

                        best_neg = neg_subset


            # No useful condition found

            if best_literal is None:
                break


            # Add best condition

            rule.append(best_literal)

            current_pos = best_pos

            current_neg = best_neg


        if len(rule) == 0:
            return None


        return rule


    # --------------------------------------------------------
    # LEARN ALL RULES
    # --------------------------------------------------------

    def learn_rules(self):

        all_rules = []


        target_values = self.data[
            self.target_column
        ].unique()


        for target_value in target_values:

            print()
            print("----------------------------------------")
            print(
                "Learning rules for:",
                target_value
            )
            print("----------------------------------------")


            positives = self.data[
                self.data[self.target_column]
                == target_value
            ].copy()


            negatives = self.data[
                self.data[self.target_column]
                != target_value
            ].copy()


            uncovered = positives.copy()


            # Learn rules until all positive
            # examples are covered

            while len(uncovered) > 0:

                rule = self.learn_one_rule(
                    uncovered,
                    negatives
                )


                if rule is None:
                    break


                all_rules.append(
                    (target_value, rule)
                )


                print(
                    self.rule_to_string(
                        rule,
                        target_value
                    )
                )


                # Find examples covered by rule

                covered = self.apply_rule(
                    uncovered,
                    rule
                )


                if len(covered) == 0:
                    break


                # Remove covered positives

                uncovered = uncovered.drop(
                    covered.index
                )


        return all_rules


    # --------------------------------------------------------
    # CONVERT RULE TO READABLE FORM
    # --------------------------------------------------------

    def rule_to_string(
        self,
        rule,
        target
    ):

        conditions = []

        for feature, value in rule:

            conditions.append(
                f"{feature} = {value}"
            )


        return (
            "IF "
            + " AND ".join(conditions)
            + f" THEN "
            + f"{self.target_column} = {target}"
        )


# ============================================================
# RUN FOIL
# ============================================================

print()
print("================================================")
print("FOIL FIRST-ORDER RULES")
print("================================================")


foil = FOIL(
    df,
    "Play"
)


foil_rules = foil.learn_rules()


# ============================================================
# FINAL RULES
# ============================================================

print()
print("================================================")
print("FINAL FOIL RULES")
print("================================================")


for target, rule in foil_rules:

    print(
        foil.rule_to_string(
            rule,
            target
        )
    )

DATASET
     Outlook Temperature Humidity    Wind Play
0      Sunny         Hot     High    Weak   No
1      Sunny         Hot     High  Strong   No
2   Overcast         Hot     High    Weak  Yes
3       Rain        Mild     High    Weak  Yes
4       Rain        Cool   Normal    Weak  Yes
5       Rain        Cool   Normal  Strong   No
6   Overcast        Cool   Normal  Strong  Yes
7      Sunny        Mild     High    Weak   No
8      Sunny        Cool   Normal    Weak  Yes
9       Rain        Mild   Normal    Weak  Yes
10     Sunny        Mild   Normal  Strong  Yes
11  Overcast        Mild     High  Strong  Yes
12  Overcast         Hot   Normal    Weak  Yes
13      Rain        Mild     High  Strong   No

CN2 CLASSIFICATION RULES
IF Outlook==Overcast THEN Play=Yes 
IF Temperature==Hot THEN Play=No 
IF Humidity==Normal AND Outlook!=Rain THEN Play=Yes 
IF Outlook==Sunny THEN Play=No 
IF Wind==Weak THEN Play=Yes 
IF Outlook==Rain THEN Play=No 
IF TRUE THEN Play=Yes 


FOIL FIRST-ORDER RULE

In [3]:
import pandas as pd
import math

# ============================================================
# 1. READ THE DATASET
# ============================================================

df = pd.read_csv("play_tennis.csv")

print("========== DATASET ==========")
print(df)
print()


# ============================================================
# 2. CN2 CLASSIFICATION RULE LEARNING
# ============================================================

# Install Orange3 if you haven't:
# pip install orange3

from Orange.data import Table, Domain, DiscreteVariable
from Orange.classification.rules import CN2Learner


# Create input attributes
attributes = []

for column in df.columns[:-1]:

    values = [
        str(value)
        for value in df[column].unique()
    ]

    attributes.append(
        DiscreteVariable(
            column,
            values=values
        )
    )


# Create target variable
target_column = df.columns[-1]

target_values = [
    str(value)
    for value in df[target_column].unique()
]

target = DiscreteVariable(
    target_column,
    values=target_values
)


# Create Orange domain
domain = Domain(
    attributes,
    target
)


# Convert Pandas DataFrame to Orange Table
orange_data = Table.from_list(
    domain,
    df.astype(str).values.tolist()
)


# Train CN2
learner = CN2Learner()

classifier = learner(orange_data)


# Display CN2 rules
print("========== CN2 CLASSIFICATION RULES ==========")

for rule in classifier.rule_list:
    print(rule)

print()


# ============================================================
# 3. FOIL ALGORITHM
# ============================================================

class FOIL:

    def __init__(self, data, target_column):

        self.data = data
        self.target_column = target_column


    # --------------------------------------------------------
    # FOIL INFORMATION GAIN
    # --------------------------------------------------------

    def foil_gain(self, p, n, p_new, n_new):

        if p == 0 or p_new == 0:
            return 0

        old_probability = p / (p + n)

        new_probability = p_new / (p_new + n_new)

        if old_probability == 0 or new_probability == 0:
            return 0

        return p_new * (
            math.log2(new_probability)
            - math.log2(old_probability)
        )


    # --------------------------------------------------------
    # APPLY A RULE TO DATA
    # --------------------------------------------------------

    def apply_rule(self, data, rule):

        result = data.copy()

        for feature, value in rule:

            result = result[
                result[feature] == value
            ]

        return result


    # --------------------------------------------------------
    # LEARN ONE RULE
    # --------------------------------------------------------

    def learn_one_rule(self, positives, negatives):

        rule = []

        current_pos = positives.copy()
        current_neg = negatives.copy()

        features = [
            column
            for column in self.data.columns
            if column != self.target_column
        ]


        # Continue adding conditions until
        # negative examples are eliminated

        while len(current_neg) > 0:

            best_literal = None
            best_score = -float("inf")

            best_pos = None
            best_neg = None


            # Try every possible condition

            for feature in features:

                for value in self.data[feature].unique():

                    literal = (feature, value)


                    # Don't use the same condition twice

                    if literal in rule:
                        continue


                    # Examples satisfying this condition

                    pos_subset = current_pos[
                        current_pos[feature] == value
                    ]

                    neg_subset = current_neg[
                        current_neg[feature] == value
                    ]


                    # Condition must cover
                    # at least one positive example

                    if len(pos_subset) == 0:
                        continue


                    # Calculate FOIL information gain

                    score = self.foil_gain(
                        len(current_pos),
                        len(current_neg),
                        len(pos_subset),
                        len(neg_subset)
                    )


                    # Select the best condition

                    if score > best_score:

                        best_score = score

                        best_literal = literal

                        best_pos = pos_subset

                        best_neg = neg_subset


            # No useful condition found

            if best_literal is None:
                break


            # Add the best condition to the rule

            rule.append(best_literal)

            current_pos = best_pos

            current_neg = best_neg


        if len(rule) == 0:
            return None


        return rule


    # --------------------------------------------------------
    # LEARN ALL RULES
    # --------------------------------------------------------

    def learn_rules(self):

        all_rules = []


        target_values = self.data[
            self.target_column
        ].unique()


        # Learn rules for every class

        for target_value in target_values:

            print()
            print("----------------------------------------")
            print(
                "Learning rules for:",
                target_value
            )
            print("----------------------------------------")


            # Positive examples

            positives = self.data[
                self.data[self.target_column]
                == target_value
            ].copy()


            # Negative examples

            negatives = self.data[
                self.data[self.target_column]
                != target_value
            ].copy()


            uncovered = positives.copy()


            # Continue until all positive examples
            # have been covered

            while len(uncovered) > 0:

                rule = self.learn_one_rule(
                    uncovered,
                    negatives
                )


                if rule is None:
                    break


                all_rules.append(
                    (target_value, rule)
                )


                print(
                    self.rule_to_string(
                        rule,
                        target_value
                    )
                )


                # Find positive examples covered
                # by this rule

                covered = self.apply_rule(
                    uncovered,
                    rule
                )


                if len(covered) == 0:
                    break


                # Remove covered examples

                uncovered = uncovered.drop(
                    covered.index
                )


        return all_rules


    # --------------------------------------------------------
    # DISPLAY RULE
    # --------------------------------------------------------

    def rule_to_string(self, rule, target):

        conditions = []

        for feature, value in rule:

            conditions.append(
                f"{feature} = {value}"
            )


        return (
            "IF "
            + " AND ".join(conditions)
            + f" THEN "
            + f"{self.target_column} = {target}"
        )


# ============================================================
# 4. RUN FOIL
# ============================================================

print()
print("========== FOIL FIRST-ORDER RULES ==========")

foil = FOIL(
    df,
    target_column
)

foil_rules = foil.learn_rules()


# ============================================================
# 5. DISPLAY FINAL RULES
# ============================================================

print()
print("========== FINAL FOIL RULES ==========")

for target, rule in foil_rules:

    print(
        foil.rule_to_string(
            rule,
            target
        )
    )

========== DATASET ==========
     Outlook Temperature Humidity    Wind Play
0      Sunny         Hot     High    Weak   No
1      Sunny         Hot     High  Strong   No
2   Overcast         Hot     High    Weak  Yes
3       Rain        Mild     High    Weak  Yes
4       Rain        Cool   Normal    Weak  Yes
5       Rain        Cool   Normal  Strong   No
6   Overcast        Cool   Normal  Strong  Yes
7      Sunny        Mild     High    Weak   No
8      Sunny        Cool   Normal    Weak  Yes
9       Rain        Mild   Normal    Weak  Yes
10     Sunny        Mild   Normal  Strong  Yes
11  Overcast        Mild     High  Strong  Yes
12  Overcast         Hot   Normal    Weak  Yes
13      Rain        Mild     High  Strong   No

========== CN2 CLASSIFICATION RULES ==========
IF Outlook==Overcast THEN Play=Yes 
IF Temperature==Hot THEN Play=No 
IF Humidity==Normal AND Outlook!=Rain THEN Play=Yes 
IF Outlook==Sunny THEN Play=No 
IF Wind==Weak THEN Play=Yes 
IF Outlook==Rain THEN Play=No 
IF